# RELATIVE SIGNAL FREQUENCY CALCULATION
In order to automatically generate the M matrix at any given signal input frequency, we need to come up with a way to automatically compute all the relative idler frequencies given a specified mothertone/mode.

We can specify the graph with a `nodes` list and an `edges` list of dictionaries.

In [2]:
def node(
    f0: float = 6.0,   # natural frequency
    Bint: float = 0.0, # internal loss rate  
    Bext: float = 1.0, # external loss rate
    conjugated: bool = False, # whether the node is conjugated
    label: str = 'A', # label for the node
    Bscale: float = 1e-3, # relative scale factor for the loss vs frequency units
) -> dict:
    """
    Create a node dictionary for the graph.
    Note that the units are arbs and just have to be consistent.
    BSCALE is the relative scale factor for the loss vs frequency units, i.e., 
    if BSCALE = 1e-3, and f0 is chosen to be in GHz, then Bint and Bext are in MHz.
    """
    return {
        "f0": f0,
        "Bint": Bint,
        "Bext": Bext,
        "conjugated": conjugated,
        "label": label,
        "Bscale": Bscale
    }

def edge(
    jk: (int, int) = (0, 1), # tuple of node indices
    fp: float = 1.0, # frequency of the pump tone
    mag: float = 1.0, # magnitude of the pump tone in same arb units as BSCALE
    phase: float = 0.0, # [degrees] phase of the pump tone 
    Bscale: float = 1e-3, # relative scale factor for the coupling vs frequency units
):
    """
    Create an edge dictionary for the graph.

    magnitude is the pump tone magnitude in same arb units as BSCALE
    phase is the pump tone phase in degrees
    BSCALE is the relative scale factor for the coupling vs frequency units
    """
    return {
        "jk": jk,
        "fp": fp,
        "mag": mag,
        "phase": phase,
        "Bscale": Bscale
    }

def graph(
    nodes: list[dict],
    edges: list[dict]
) -> dict:
    """
    Create a graph dictionary for the graph.
    """
    return {
        "nodes": nodes,
        "edges": edges
    }


# EXAMPLE 4 mode square graph + 1 dingleberry mode

nodes = [
    node(f0=6.0, Bint=0.0, Bext=1.0, label='A'),
    node(f0=7.2, Bint=0.0, Bext=1.0, label='B'),
    node(f0=8.1, Bint=0.0, Bext=1.0, label='C', conjugated=True),
    node(f0=9.4, Bint=0.0, Bext=1.0, label='D', conjugated=True),
    node(f0=4.23, Bint=0.0, Bext=1.0, label='E'),
]

edges = [ # with some intentional detunings
    edge(jk=(0, 1), fp=1.2, mag=1.0, phase=0.0, Bscale=1e-3),
    edge(jk=(1, 2), fp=15.3, mag=1.0, phase=0.0, Bscale=1e-3),
    edge(jk=(2, 3), fp=1.3, mag=1.0, phase=0.0, Bscale=1e-3),
    edge(jk=(3, 0), fp=3.45, mag=1.0, phase=0.0, Bscale=1e-3),
    edge(jk=(3, 4), fp=13.6, mag=0.3, phase=0.0, Bscale=1e-3),
]

g = graph(nodes, edges)



In [ ]:
# specify mothertone
mothertone = 'B'

# get index of mothertone in nodes list
mothertone_idx = [node['label'] for node in nodes].index(mothertone)


# Build a list of all of the idler frequencies relative to the mothertone
idler_freqs = np.zeros(len(nodes))

# traverse the graph by starting at the mothertone and following the edges

# start with the mothertone
current_node = mothertone_idx
consumed_edges = []

while len(consumed_edges) < len(edges):
   for edge in edges:
    j,k = edge['jk']
    if j == current_node:
        # then the current node connects to node k and we have to add or subtract the 
        # pump frequency from the current node frequency to get the idler frequency. 
        # The sign depends on whether the two nodes have the same conjugation or not.

        idler_freqs[k] = edge['fp'] - nodes[k]['f0']
    elif k == current_node:
        idler_freqs[j] = edge['fp'] - nodes[j]['f0']

print(idler_freqs)

[{'jk': (0, 1), 'fp': 1.2, 'mag': 1.0, 'phase': 0.0, 'Bscale': 0.001}, {'jk': (0, 1), 'fp': 1.2, 'mag': 1.0, 'phase': 0.0, 'Bscale': 0.001}, {'jk': (0, 1), 'fp': 1.2, 'mag': 1.0, 'phase': 0.0, 'Bscale': 0.001}, {'jk': (0, 1), 'fp': 1.2, 'mag': 1.0, 'phase': 0.0, 'Bscale': 0.001}, {'jk': (0, 1), 'fp': 1.2, 'mag': 1.0, 'phase': 0.0, 'Bscale': 0.001}]
